# Lab | Transformers
## Route every message to the right team

Yesterday you built a semantic search for a digital bank's support messages. The bank now wants to go one step further: **route** each new message automatically to the team that handles its intent, before any person reads it.

The support lead has picked the ten intents that agents mix up most often: card delivery, lost and compromised cards, PINs, failed top-ups and exchange rates. If a model can tell these apart, the rest will be easy. Your job is to compare three ways of doing it, and to recommend one:

+ a **zero-shot** model, which needs no labelled examples at all;
+ a **fine-tuned DistilBERT**, trained on the bank's labelled messages;
+ yesterday's **semantic search**, used as a classifier: the intent of the most similar past message.

**The data** is the Banking77 dataset from yesterday's lab, in `data/banking_train.csv` and `data/banking_test.csv`, filtered to the ten intents.

**This lab practises** using and fine-tuning pretrained transformers. You will use, from today's **Transformers and Hugging Face** lesson: `AutoTokenizer`, the zero-shot `pipeline`, `Dataset.map`, `AutoModelForSequenceClassification`, `TrainingArguments` and `Trainer`, and the comparison with the methods of Days 1 and 2.

**By the end, you can:**

1. Apply a zero-shot pipeline and fine-tune DistilBERT to classify customer messages by intent.
2. Evaluate both against a semantic-search baseline and justify which one to use.

Each challenge ends with a **checkpoint** that tells you what you should see. If your numbers are very different, fix that before moving on. Fine-tuning takes one to four minutes on a laptop CPU; on a slow computer, open the notebook in [Google Colab](https://colab.research.google.com) with a GPU runtime.

## Setup

You need `transformers`, `torch`, `datasets`, `accelerate` and `sentence-transformers`. If the import fails, install them from a terminal with `pip install transformers torch datasets accelerate sentence-transformers`. The first run downloads three models (about 500 MB in total) unless you already have them from the lesson.

In [1]:
%pip install datasets

In [2]:
import os
import logging
import warnings
os.environ["HF_HUB_VERBOSITY"] = "error"      # hide Hugging Face download notices
os.environ["TOKENIZERS_PARALLELISM"] = "false"
logging.getLogger("torch.distributed.elastic.multiprocessing.redirects").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message=".*torch.jit.script.*")

import time
import numpy as np
import pandas as pd
import torch

from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from transformers.utils import logging as transformers_logging
transformers_logging.set_verbosity_error()
transformers_logging.disable_progress_bar()
import datasets
datasets.disable_progress_bars()
from sentence_transformers import SentenceTransformer

from sklearn.metrics import accuracy_score, classification_report

pd.set_option("display.max_colwidth", 90)
_ = torch.manual_seed(0)

In [11]:
# google.colab.drive.mount is disabled because credential propagation is unavailable in this environment.
# The dataset is already loaded locally from '../data/' in the next cells, so mounting drive is not required.
#
# from google.colab import drive
# drive.mount('/content/drive')

In [18]:
print(os.listdir("/content"))

['.config', 'drive', 'data', 'sample_data']


In [20]:
intents = ["card_arrival", "card_delivery_estimate", "lost_or_stolen_card", "compromised_card", "card_not_working",
           "pin_blocked", "change_pin", "top_up_failed", "top_up_reverted", "exchange_rate"]

train = pd.read_csv("data/banking_train.csv")
test = pd.read_csv("data/banking_test.csv")
train = train[train["category"].isin(intents)].reset_index(drop=True)
test = test[test["category"].isin(intents)].reset_index(drop=True)
print(train.shape, test.shape)
train.sample(5, random_state=0)

(1185, 2) (400, 2)


,text,category
833,What do I do if I detect fraudulent use on my car?,compromised_card
567,I lost my card. Can you help me?,lost_or_stolen_card
950,Can my PIN changed remotely without visiting a bank?,change_pin
922,I need to make my card PIN a different number,change_pin
362,how long does it take to get my card i am in a rush,card_delivery_estimate


**Checkpoint:** `(1185, 2) (400, 2)`: 1,185 past messages to learn from, and 400 new messages (40 per intent) to test on.

## Challenge 1: look at the messages as tokens

Before choosing a model, check what the model will actually read.

1. Load the tokenizer of `distilbert/distilbert-base-uncased` and print the tokens of three messages: `"My card hasn't arrived yet"`, `"Someone used my card in Bulgaria!"` and `"Why did my top-up get reverted?"`.
2. Count the tokens of every training message (`len(tokenizer(text)["input_ids"])`) and print the `describe()` of the counts.

In [21]:
checkpoint = "distilbert/distilbert-base-uncased"

# 1. Tokenizer and the tokens of three messages
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
print("Vocabulary size:", tokenizer.vocab_size)

print(tokenizer.tokenize("My card hasn't arrived yet"))
print(tokenizer.tokenize("Someone used my card in Bulgaria!"))
print(tokenizer.tokenize("Why did my top-up get reverted?"))
# 2. Number of tokens per training message
messages = [
    "My card hasn't arrived yet",
    "Someone used my card in Bulgaria!",
    "Why did my top-up get reverted?"
]

token_counts = [
    len(tokenizer(text)["input_ids"])
    for text in messages
]

df = pd.DataFrame({
    "message": messages,
    "token_count": token_counts
})

print(df)
print(df["token_count"].describe())


Vocabulary size: 30522
['my', 'card', 'hasn', "'", 't', 'arrived', 'yet']
['someone', 'used', 'my', 'card', 'in', 'bulgaria', '!']
['why', 'did', 'my', 'top', '-', 'up', 'get', 'reverted', '?']
                             message  token_count
0         My card hasn't arrived yet            9
1  Someone used my card in Bulgaria!            9
2    Why did my top-up get reverted?           11
count     3.000000
mean      9.666667
std       1.154701
min       9.000000
25%       9.000000
50%       9.000000
75%      10.000000
max      11.000000
Name: token_count, dtype: float64


**Checkpoint:** `hasn't` becomes `hasn`, `'` and `t`, and `top-up` becomes `top`, `-` and `up`. The messages are short: a median of 13 tokens and a mean of about 15, and the longest has 66, including `[CLS]` and `[SEP]`.

**Question:** you will fine-tune with `max_length=64`. How many training messages does that cut, and why is a short `max_length` worth having?

Using max_length=64 truncates all training messages containing more than 64 tokens, using sum(count > 64 for count in token_counts) the exact number of phrases to be truncated can be calculated. A short 'max_length' makes training faster and reduces memory consumption. However, it may remove useful information from longer messages, so it is importatn to check how many phrases are affected before.

after applying the code below, only one phrase from the train set was longer than 64 tokens

In [25]:
tokenizer(
    text,
    truncation=True,
    max_length=64
)

Message: I am still waiting on my card?
Tokens: [101, 1045, 2572, 2145, 3403, 2006, 2026, 4003, 1029, 102]
Token count: 10


In [26]:
# Count tokens for every training message
token_counts = [
    len(tokenizer(text)["input_ids"])
    for text in train["text"]
]

# Count messages exceeding 64 tokens
cut_messages = sum(count > 64 for count in token_counts)

print("Total training messages:", len(train))
print("Messages exceeding 64 tokens:", cut_messages)
print("Percentage truncated:", cut_messages / len(train) * 100)

Total training messages: 1185
Messages exceeding 64 tokens: 1
Percentage truncated: 0.08438818565400844


## Challenge 2: zero-shot routing

The zero-shot model needs no training, only a description of each intent in plain English. The quality of those descriptions matters: the model compares each message with each description.

1. Complete the dictionary `descriptions` with a short description for the last three intents. Seven are given.
2. Create the zero-shot pipeline with the model `MoritzLaurer/deberta-v3-xsmall-zeroshot-v1.1-all-33`.
3. Run it on all the test messages, with `candidate_labels=list(descriptions.values())` and `batch_size=16`, and time it.
4. Turn the top label of each result back into an intent and compute the accuracy.

In [29]:
descriptions = {
    "card_arrival": "my new card has not arrived",
    "card_delivery_estimate": "how long card delivery takes",
    "lost_or_stolen_card": "my card was lost or stolen",
    "compromised_card": "someone else used my card",
    "card_not_working": "my card does not work",
    "pin_blocked": "my PIN is blocked",
    "change_pin": "how to change my PIN",
    "top_up_reverted": "Why did my top-up get reverted?",
    "top_up_failed": "Top-up is not working",
    "exchange_rate": "Will my money be of equal value when I travel abroad?"
    }

# 2. Zero-shot pipeline
zero_shot = pipeline("zero-shot-classification", model="MoritzLaurer/deberta-v3-xsmall-zeroshot-v1.1-all-33", device=0)

candidate_labels = list(descriptions.values())

description_to_intent = {
    description: intent
    for intent, description in descriptions.items()
}

# 3. Predictions for the test messages, timed

test_messages = test["text"].tolist()
true_intents = test["category"].tolist()

torch.cuda.synchronize()
start = time.perf_counter()

results = zero_shot(
    test_messages,
    candidate_labels=candidate_labels,
    batch_size=16
)

torch.cuda.synchronize()
elapsed = time.perf_counter() - start

print(f"Time taken: {elapsed_time:.2f} seconds")



# 4. Intents and accuracy
predicted_intents = [
    description_to_intent[result["labels"][0]]
    for result in results
]

accuracy = accuracy_score(true_intents, predicted_intents)

print(f"Number of test messages: {len(test_messages)}")
print(f"Execution time: {elapsed:.2f} seconds")
print(f"Zero-shot accuracy: {accuracy:.2%}")

Time taken: 9.15 seconds
Number of test messages: 400
Execution time: 7.51 seconds
Zero-shot accuracy: 63.75%


In [31]:
#Examining incorrect predictions
results_df = pd.DataFrame({
    "message": test_messages,
    "actual_intent": true_intents,
    "predicted_intent": predicted_intents
})

errors = results_df[
    results_df["actual_intent"] != results_df["predicted_intent"]
]

print("Incorrect predictions:", len(errors))
display(errors.head(20))

Incorrect predictions: 145


,message,actual_intent,predicted_intent
0,How do I locate my card?,card_arrival,lost_or_stolen_card
3,Is there a way to know when my card will arrive?,card_arrival,card_delivery_estimate
4,My card has not arrived yet.,card_arrival,card_delivery_estimate
8,still waiting on that card,card_arrival,card_delivery_estimate
9,Is it normal to have to wait over a week for my new card?,card_arrival,card_delivery_estimate
10,How do I track my card?,card_arrival,lost_or_stolen_card
11,How long does a card delivery take?,card_arrival,card_delivery_estimate
14,I am still waiting for my card after 1 week. Is this ok?,card_arrival,card_delivery_estimate
15,"I have been waiting longer than expected for my bank card, could you provide informati...",card_arrival,card_delivery_estimate
16,I've been waiting longer than expected for my card.,card_arrival,card_delivery_estimate


**Checkpoint:** the 400 messages take a few seconds to a minute, and the accuracy is about 0.76 with these descriptions (your own descriptions for the last three intents will move it a few points).

In [32]:
# 5. Which intents does the zero-shot model confuse? Show the crosstab of true and predicted intents.
crosstab = pd.crosstab(
    results_df["actual_intent"],
    results_df["predicted_intent"]
)

print("Crosstab:")
display(crosstab)

Crosstab:


predicted_intent,card_arrival,card_delivery_estimate,card_not_working,change_pin,compromised_card,exchange_rate,lost_or_stolen_card,pin_blocked,top_up_failed,top_up_reverted
actual_intent,,,,,,,,,,
card_arrival,18,15,0,0,1,0,5,1,0,0
card_delivery_estimate,18,19,1,0,0,0,0,1,1,0
card_not_working,0,0,40,0,0,0,0,0,0,0
change_pin,0,0,0,40,0,0,0,0,0,0
compromised_card,1,0,2,0,26,0,10,1,0,0
exchange_rate,1,0,1,1,3,9,1,2,21,1
lost_or_stolen_card,3,0,3,0,2,0,31,0,1,0
pin_blocked,0,0,6,4,0,0,0,30,0,0
top_up_failed,0,0,2,0,0,0,0,0,33,5


**Question:** which intents does the zero-shot model handle well, and which does it confuse? Why is that hard to fix with better descriptions?

The zero-shot model performs well with clearly defined intents. change_pin and card_not_working, achieving 100% recall for both. However, it struggles with overlapping categories, especially card_arrival versus card_delivery_estimate, top_up_reverted versus top_up_failed, and compromised_card versus lost_or_stolen_card. It also performs poorly on exchange_rate, frequently predicting top_up_failed instead.

I think these problems are difficult to solve through better descriptions alone because several intents are semantically similar. Zero-shot classification cannot learn the dataset's specific distinctions from labeled examples. Fine-tuning should help the model recognize these differences more accurately.

## Challenge 3: fine-tune DistilBERT

Now use the bank's 1,185 labelled messages. Follow the steps of section 6 of the lesson; the skeleton below has the structure, and you fill in the `TODO` parts.

1. Map every intent to a number (`label2id`) and back (`id2label`).
2. Write `to_dataset(df)`: build a `datasets.Dataset` with the columns `text` and `label`, and `map` the tokenizer on it with `truncation=True, max_length=64`.
3. Load `AutoModelForSequenceClassification` with 10 labels, `id2label` and `label2id`.
4. Train for 3 epochs with a batch size of 16 and a learning rate of `5e-5`, evaluating after every epoch.

In [37]:
# 1. Intents as numbers
labels = sorted(intents)
label2id = {label: i for i, label in enumerate(labels)}   # TODO
id2label = {i: label for label, i in label2id.items()}   # TODO

# 2. Datasets
from datasets import Dataset

def to_dataset(df):
    data = Dataset.from_dict({
        "text": df["text"].tolist(),
        "label": df["category"].map(label2id).tolist()
    })

    data = data.map(
        lambda batch: tokenizer(
            batch["text"],
            truncation=True,
            max_length=64
        ),
        batched=True
    )

    return data

train_data = to_dataset(train)
test_data = to_dataset(test)

# 3. Model with a new classification layer
model_name = "distilbert-base-uncased"
classifier = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=10,
    id2label=id2label,
    label2id=label2id
)

def compute_metrics(prediction):
    predicted = prediction.predictions.argmax(axis=-1)
    return {"accuracy": accuracy_score(prediction.label_ids, predicted)}

# 4. Training
from transformers import DataCollatorWithPadding

args = TrainingArguments(
    output_dir="intent-classifier",
    # TODO: epochs, batch size, learning rate, evaluation after every epoch
    logging_strategy="epoch",
    save_strategy="no",
    report_to="none",
    disable_tqdm=True,
    dataloader_pin_memory=False,
    seed=0,
)
trainer = Trainer(
    model=classifier,
    args=args,
    train_dataset=train_data,
    eval_dataset=test_data,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer)
)

start = time.time()
trainer.train()
print(f"Training took {time.time() - start:.0f} seconds")

{'loss': '1.093', 'grad_norm': '1.822', 'learning_rate': '3.345e-05', 'epoch': '1'}
{'loss': '0.2362', 'grad_norm': '1.529', 'learning_rate': '1.678e-05', 'epoch': '2'}
{'loss': '0.09778', 'grad_norm': '25', 'learning_rate': '1.119e-07', 'epoch': '3'}
{'train_runtime': '29.81', 'train_samples_per_second': '119.3', 'train_steps_per_second': '15', 'train_loss': '0.4756', 'epoch': '3'}
Training took 30 seconds


In [38]:
# 5. Show the loss, the test loss and the test accuracy of every epoch (trainer.state.log_history)
trainer.state.log_history

[{'loss': 1.0928521892368392,
  'grad_norm': 1.8217226266860962,
  'learning_rate': 3.3445190156599555e-05,
  'epoch': 1.0,
  'step': 149},
 {'loss': 0.2362004478505794,
  'grad_norm': 1.5286709070205688,
  'learning_rate': 1.6778523489932888e-05,
  'epoch': 2.0,
  'step': 298},
 {'loss': 0.09778075250203178,
  'grad_norm': 24.999099731445312,
  'learning_rate': 1.1185682326621924e-07,
  'epoch': 3.0,
  'step': 447},
 {'train_runtime': 29.8074,
  'train_samples_per_second': 119.266,
  'train_steps_per_second': 14.996,
  'total_flos': 27005468677200.0,
  'train_loss': 0.4756111298631502,
  'epoch': 3.0,
  'step': 447}]

In [42]:
test_results = trainer.evaluate(test_data)

print(f"Test loss: {test_results['eval_loss']:.4f}")
print(f"Test accuracy: {test_results['eval_accuracy']:.2%}")

{'eval_loss': '0.1991', 'eval_accuracy': '0.9475', 'eval_runtime': '1.683', 'eval_samples_per_second': '237.6', 'eval_steps_per_second': '29.7', 'epoch': '3'}
Test loss: 0.1991
Test accuracy: 94.75%


**Checkpoint:** training takes one to four minutes on a CPU (a few seconds on a GPU). The test accuracy is about 0.92 after the first epoch and about 0.94 after the third, and both the training loss and the test loss fall at every epoch.

**Question:** the classification layer started with random weights. Why does the model still reach a good accuracy after a single pass over 1,185 messages?

Although the classification layer has random weights, DistilBERT's Transformer layers have already learned useful language representations during pre-training. Fine-tuning therefore does not require the model to learn language from scratch. It only needs to adapt its existing knowledge to distinguish the 10 banking intents. Since the categories are relatively well-defined, even 1,185 labeled examples can provide enough information to achieve good accuracy after a single epoch. Both the classification layer and the pre-trained layers are updated during fine-tuning.

## Challenge 4: compare with yesterday's semantic search

Yesterday's semantic search can also classify: give a new message the intent of the most similar past message. Use it as the baseline.

1. Encode the training and test messages with `SentenceTransformer("all-MiniLM-L6-v2")` and `normalize_embeddings=True`.
2. For each test message, take the intent of the training message with the highest dot product, and compute the accuracy.
3. Get the predictions of the fine-tuned model on the test set (`trainer.predict(test_data)`, then `argmax` and `id2label`).
4. Build a DataFrame with one row per method (zero-shot, semantic search, fine-tuned DistilBERT), with the accuracy and the number of labelled messages each one used.

In [44]:
# 1. Embeddings
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

train_embeddings = embedding_model.encode(
    train["text"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)

test_embeddings = embedding_model.encode(
    test["text"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)

# 2. Semantic search as a classifier
similarities = test_embeddings @ train_embeddings.T
nearest_indices = similarities.argmax(axis=1)

#getting the corresponding intents
semantic_predictions = train["category"].iloc[
    nearest_indices
].tolist()
#calculating accuracy
from sklearn.metrics import accuracy_score

semantic_accuracy = accuracy_score(
    test["category"],
    semantic_predictions
)

print(f"Semantic search accuracy: {semantic_accuracy:.2%}")

# 3. Fine-tuned predictions
predictions = trainer.predict(test_data)

# Get the predicted class IDs
predicted_ids = predictions.predictions.argmax(axis=-1)

# Convert numerical IDs back to intent names
finetuned_predictions = [
    id2label[int(i)] for i in predicted_ids
]

# Calculate accuracy
finetuned_accuracy = accuracy_score(
    test["category"],
    finetuned_predictions
)

print(f"Fine-tuned DistilBERT accuracy: {finetuned_accuracy:.2%}")

# 4. Comparison table
# Zero-shot accuracy
zero_shot_accuracy = accuracy_score(
    test["category"],
    predicted_intents
)

# Create comparison DataFrame
comparison = pd.DataFrame({
    "Method": [
        "Zero-shot",
        "Semantic search",
        "Fine-tuned DistilBERT"
    ],
    "Accuracy": [
        zero_shot_accuracy,
        semantic_accuracy,
        finetuned_accuracy
    ],
    "Labelled messages used": [
        0,
        len(train),
        len(train)
    ]
})

display(comparison)

Batches:   0%|          | 0/38 [00:00<?, ?it/s]

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

Semantic search accuracy: 91.50%
Fine-tuned DistilBERT accuracy: 94.75%


,Method,Accuracy,Labelled messages used
0,Zero-shot,0.6375,0
1,Semantic search,0.9150,1185
2,Fine-tuned DistilBERT,0.9475,1185


**Checkpoint:** zero-shot about 0.76, semantic search about 0.92, fine-tuned DistilBERT about 0.94. Your fine-tuned accuracy can differ by a point or two from run to run and from computer to computer.

In [46]:
# 5. Print the classification report of the fine-tuned model, and the 6 most frequent (true, predicted) mistakes

print(classification_report(
    test["category"],
    finetuned_predictions
))

# DataFrame with actual and predicted intents
errors_df = pd.DataFrame({
    "true": test["category"].tolist(),
    "predicted": finetuned_predictions
})

# Keeping only incorrect predictions
errors_df = errors_df[
    errors_df["true"] != errors_df["predicted"]
]

# Count and display the 6 most frequent mistakes
top_6_errors = (
    errors_df
    .groupby(["true", "predicted"])
    .size()
    .sort_values(ascending=False)
    .head(6)
)

print(top_6_errors)

                        precision    recall  f1-score   support

          card_arrival       0.88      0.90      0.89        40
card_delivery_estimate       0.92      0.90      0.91        40
      card_not_working       0.95      0.97      0.96        40
            change_pin       0.95      1.00      0.98        40
      compromised_card       1.00      0.95      0.97        40
         exchange_rate       1.00      1.00      1.00        40
   lost_or_stolen_card       0.93      0.95      0.94        40
           pin_blocked       0.97      0.93      0.95        40
         top_up_failed       0.91      0.97      0.94        40
       top_up_reverted       0.97      0.90      0.94        40

              accuracy                           0.95       400
             macro avg       0.95      0.95      0.95       400
          weighted avg       0.95      0.95      0.95       400

true                    predicted             
card_delivery_estimate  card_arrival              4
to

**Question:** where does the fine-tuned model still make mistakes? Are they model errors, or messages that a person would also find ambiguous?

The fine-tuned DistilBERT achieves 95% accuracy but still makes mistakes between semantically similar intents, particularly card_arrival and card_delivery_estimate, top_up_reverted and top_up_failed, and pin_blocked and change_pin.

Some of these errors may come from ambiguity in customer messages. For example, a customer asking why their top-up did not work might be describing either a failed or a reverted transaction.

However, not all mistakes can be attributed to ambiguity. If a message clearly expresses one intent but the model predicts another, this can be a model error. To distinguish the two, an inspection of the misclassified messages individually is needed.

Overall, the results demonstrate that fine-tuning significantly improves the model's ability to distinguish similar intents, although some semantic overlap remains challenging.

## Challenge 5: route new messages and recommend a method

1. Call `trainer.model.eval()`, then wrap the fine-tuned model in a `text-classification` pipeline (`model=trainer.model, tokenizer=tokenizer`).
2. Route five new messages: four that you write yourself, one for each of four different intents, plus `"I don't know what happened, the app just says error"`. Print each message with its predicted intent and score.

In [47]:
# 1. Pipeline
trainer.model.eval()

from transformers import pipeline

intent_classifier = pipeline(
    "text-classification",
    model=trainer.model,
    tokenizer=tokenizer,
    device=0
)

# 2. Five new messages

messages = [
    "I ordered my bank card two weeks ago, but it still hasn't arrived.",
    "I noticed a payment in Spain that I definitely did not authorize.",
    "I entered my PIN incorrectly several times and now my card is blocked.",
    "The money I added to my account was returned to my bank.",
    "I don't know what happened, the app just says error"
]

#Predicting the results
predictions = intent_classifier(
    messages,
    truncation=True,
    max_length=64
)

for message, prediction in zip(messages, predictions):
    print(f"Message: {message}")
    print(f"Predicted intent: {prediction['label']}")
    print(f"Score: {prediction['score']:.3f}")
    print("-" * 50)

Message: I ordered my bank card two weeks ago, but it still hasn't arrived.
Predicted intent: card_arrival
Score: 0.987
--------------------------------------------------
Message: I noticed a payment in Spain that I definitely did not authorize.
Predicted intent: card_not_working
Score: 0.420
--------------------------------------------------
Message: I entered my PIN incorrectly several times and now my card is blocked.
Predicted intent: pin_blocked
Score: 0.987
--------------------------------------------------
Message: The money I added to my account was returned to my bank.
Predicted intent: top_up_reverted
Score: 0.972
--------------------------------------------------
Message: I don't know what happened, the app just says error
Predicted intent: pin_blocked
Score: 0.570
--------------------------------------------------


**Checkpoint:** the messages that clearly belong to one intent get the right one with a high score (0.9 or more for the card that has not arrived and for the blocked PIN). Messages that are vague, or that sit between intents, get a lower score: in the reference solution, the unknown payment in a shop goes to `card_not_working` with a score of only 0.26, and the app error message, which belongs to none of the ten intents, is still forced into one of them.

**Question:** the bank asks you which method to put in production. Write a short recommendation (4 to 6 sentences): which method, why, and what should happen to messages like the last one.

I would recommend deploying the fine-tuned DistilBERT model, as it achieved 95% test accuracy and outperformed the zero-shot classifier. Fine-tuning allows it to distinguish similar banking intents more effectively, although its incorrect classification of an unauthorized payment shows that errors remain. Ambiguous messages, such as "I don't know what happened, the app just says error," should trigger a request for clarification rather than an automatic classification. In production, I would implement a confidence threshold and an option to escalate uncertain or potentially sensitive cases to human support, while recognizing that even confident predictions can be wrong.

## Extra: how many labels do you need?

Labelling messages costs the bank time. Find out how the fine-tuned model and the semantic search behave with fewer labels.

1. Write a function that keeps `n` random training messages per intent (`train.groupby("category").sample(n, random_state=0)`).
2. For `n` in 10 and 40, fine-tune a fresh DistilBERT on that subset (same settings as Challenge 3, but 6 epochs for `n=10` because there are so few batches) and compute the test accuracy. Run the semantic search with the same subset as the past messages.
3. Show the results in a table, and add the numbers of Challenge 4 for the full training set.

In [48]:
# Your code here
#1
def sample_training_data(n):
    return (
        train.groupby("category")
        .sample(n=n, random_state=0)
        .reset_index(drop=True)
    )

train_10 = sample_training_data(10)
train_40 = sample_training_data(40)

print("10 per intent:", train_10.shape)
print("40 per intent:", train_40.shape)

10 per intent: (100, 2)
40 per intent: (400, 2)


In [49]:
#2
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score
import numpy as np
import torch

model_name = "distilbert-base-uncased"

def run_finetuning(subset, n, epochs):

    # Convert subset into tokenized dataset
    subset_data = to_dataset(subset)

    # Load a fresh model
    torch.manual_seed(0)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=10,
        id2label=id2label,
        label2id=label2id
    )

    # Configure training
    args = TrainingArguments(
        output_dir=f"intent-classifier-{n}",
        num_train_epochs=epochs,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        learning_rate=5e-5,
        eval_strategy="epoch",
        logging_strategy="epoch",
        save_strategy="no",
        report_to="none",
        disable_tqdm=True,
        dataloader_pin_memory=False,
        seed=0
    )

    # Create trainer
    subset_trainer = Trainer(
        model=model,
        args=args,
        train_dataset=subset_data,
        eval_dataset=test_data,
        compute_metrics=compute_metrics,
        processing_class=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer)
    )

    # Train
    subset_trainer.train()

    # Evaluate
    predictions = subset_trainer.predict(test_data)
    predicted_ids = predictions.predictions.argmax(axis=-1)

    predicted_intents = [
        id2label[int(i)] for i in predicted_ids
    ]

    accuracy = accuracy_score(
        test["category"],
        predicted_intents
    )

    return accuracy

In [51]:
#3
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

def run_semantic_search(subset):

    # Encode the training subset
    subset_embeddings = embedding_model.encode(
        subset["text"].tolist(),
        normalize_embeddings=True
    )

    # Encode test messages
    test_embeddings = embedding_model.encode(
        test["text"].tolist(),
        normalize_embeddings=True
    )

    # Compute similarities
    similarities = test_embeddings @ subset_embeddings.T

    # Find closest training message
    nearest_indices = similarities.argmax(axis=1)

    # Predict its intent
    predictions = subset["category"].iloc[
        nearest_indices
    ].tolist()

    # Calculate accuracy
    return accuracy_score(test["category"], predictions)

    results = []

for n in [10, 40]:

    subset = sample_training_data(n)

    # 6 epochs for n=10, otherwise 3
    epochs = 6 if n == 10 else 3

    # Fine-tuned DistilBERT
    ft_accuracy = run_finetuning(subset, n, epochs)

    # Semantic search
    semantic_acc = run_semantic_search(subset)

    results.append({
        "Training messages": len(subset),
        "Messages per intent": n,
        "Fine-tuned DistilBERT": ft_accuracy,
        "Semantic search": semantic_acc
    })

    print(f"\nResults for n={n}")
    print(f"Fine-tuned accuracy: {ft_accuracy:.2%}")
    print(f"Semantic search accuracy: {semantic_acc:.2%}")

{'loss': '2.269', 'grad_norm': '4.2', 'learning_rate': '4.286e-05', 'epoch': '1'}
{'eval_loss': '2.197', 'eval_accuracy': '0.2325', 'eval_runtime': '0.3843', 'eval_samples_per_second': '1041', 'eval_steps_per_second': '65.05', 'epoch': '1'}
{'loss': '2.096', 'grad_norm': '6.284', 'learning_rate': '3.452e-05', 'epoch': '2'}
{'eval_loss': '2', 'eval_accuracy': '0.52', 'eval_runtime': '0.3812', 'eval_samples_per_second': '1049', 'eval_steps_per_second': '65.58', 'epoch': '2'}
{'loss': '1.865', 'grad_norm': '5.654', 'learning_rate': '2.619e-05', 'epoch': '3'}
{'eval_loss': '1.813', 'eval_accuracy': '0.5425', 'eval_runtime': '0.3888', 'eval_samples_per_second': '1029', 'eval_steps_per_second': '64.3', 'epoch': '3'}
{'loss': '1.691', 'grad_norm': '5.84', 'learning_rate': '1.786e-05', 'epoch': '4'}
{'eval_loss': '1.674', 'eval_accuracy': '0.645', 'eval_runtime': '0.3943', 'eval_samples_per_second': '1015', 'eval_steps_per_second': '63.41', 'epoch': '4'}
{'loss': '1.5', 'grad_norm': '5.876', '

In [52]:
results.append({
    "Training messages": len(train),
    "Messages per intent": "Full",
    "Fine-tuned DistilBERT": finetuned_accuracy,
    "Semantic search": semantic_accuracy
})

comparison = pd.DataFrame(results)

display(
    comparison.style.format({
        "Fine-tuned DistilBERT": "{:.2%}",
        "Semantic search": "{:.2%}"
    })
)

,sequence,labels,scores,Training messages,Messages per intent,Fine-tuned DistilBERT,Semantic search
0,How do I locate my card?,"['my card was lost or stolen', 'my new card has not arrived', 'someone else used my card', 'my card does not work', 'how long card delivery takes', 'Top-up is not working', 'my PIN is blocked', 'how to change my PIN', 'Will my money be of equal value when I travel abroad?', 'Why did my top-up get reverted?']","[0.8936869502067566, 0.03996218740940094, 0.03013559617102146, 0.010035188868641853, 0.006378753576427698, 0.00626760721206665, 0.0049387551844120026, 0.003289920510724187, 0.0029721956234425306, 0.0023329032119363546]",nan,nan,nan%,nan%
1,"I still have not received my new card, I ordered over a week ago.","['my new card has not arrived', 'how long card delivery takes', 'my card does not work', 'Top-up is not working', 'someone else used my card', 'my card was lost or stolen', 'my PIN is blocked', 'how to change my PIN', 'Why did my top-up get reverted?', 'Will my money be of equal value when I travel abroad?']","[0.8449514508247375, 0.13901636004447937, 0.009827430360019207, 0.0017261954490095377, 0.0015902456361800432, 0.0013004580978304148, 0.0005549674388021231, 0.0003943001793231815, 0.0003205637913197279, 0.0003180691273882985]",nan,nan,nan%,nan%
2,I ordered a card but it has not arrived. Help please!,"['my new card has not arrived', 'how long card delivery takes', 'my card does not work', 'my card was lost or stolen', 'Top-up is not working', 'someone else used my card', 'my PIN is blocked', 'Will my money be of equal value when I travel abroad?', 'how to change my PIN', 'Why did my top-up get reverted?']","[0.4633600115776062, 0.3940173387527466, 0.09585347026586533, 0.03336551412940025, 0.0055766175501048565, 0.004596168641000986, 0.0016102588269859552, 0.0005970271886326373, 0.0005248205852694809, 0.0004988349392078817]",nan,nan,nan%,nan%
3,Is there a way to know when my card will arrive?,"['how long card delivery takes', 'my new card has not arrived', 'someone else used my card', 'my card was lost or stolen', 'my card does not work', 'my PIN is blocked', 'Top-up is not working', 'how to change my PIN', 'Will my money be of equal value when I travel abroad?', 'Why did my top-up get reverted?']","[0.678340494632721, 0.21751832962036133, 0.03393789380788803, 0.024325482547283173, 0.019056035205721855, 0.008163945749402046, 0.00663724122568965, 0.005271038971841335, 0.0040572639554739, 0.0026921832468360662]",nan,nan,nan%,nan%
4,My card has not arrived yet.,"['how long card delivery takes', 'my new card has not arrived', 'my card does not work', 'my card was lost or stolen', 'Top-up is not working', 'someone else used my card', 'my PIN is blocked', 'Will my money be of equal value when I travel abroad?', 'how to change my PIN', 'Why did my top-up get reverted?']","[0.794717013835907, 0.18259723484516144, 0.009225131012499332, 0.0032989729661494493, 0.0027780134696513414, 0.002703081350773573, 0.0024088704958558083, 0.0008439430966973305, 0.0007389813545159996, 0.0006888062926009297]",nan,nan,nan%,nan%
5,When will I get my card?,"['my new card has not arrived', 'how long card delivery takes', 'my card was lost or stolen', 'my card does not work', 'someone else used my card', 'Top-up is not working', 'my PIN is blocked', 'how to change my PIN', 'Will my money be of equal value when I travel abroad?', 'Why did my top-up get reverted?']","[0.5646384954452515, 0.3764978349208832, 0.022151483222842216, 0.011593488976359367, 0.010494319722056389, 0.005184928420931101, 0.0034946203231811523, 0.002332470379769802, 0.0020826614927500486, 0.0015296712517738342]",nan,nan,nan%,nan%
6,Do you know if there is a tracking number for the new card you sent me?,"['my new card has not arrived', 'someone else used my card', 'my card was lost or stolen', 'how long card delivery takes', 'my card does not work', 'my PIN is blocked', 'Top-up is not working', 'Will my money be of equal value when I travel abroad?',

**Conclusions:** The results demonstrate that **semantic search is more data-efficient than fine-tuning when labeled training data is limited**. With only 100 training messages, semantic search achieves 85.75% accuracy compared with 68.75% for DistilBERT, and it remains ahead with 400 messages (91.25% versus 82.25%). However, DistilBERT improves more rapidly as the training dataset grows, reaching 95% accuracy with the full 1,185 messages. This suggests that semantic search is particularly useful in low-data scenarios, while fine-tuning becomes increasingly competitive as more labeled examples become available.